# Write your first kernel in Triton
This notebook implements the vector addition kernel detailed in Chapter 2 of the manuscript. It introduces the basics of the Triton JIT compiler, the block-oriented programming model, masking, and benchmarking operations against pure Python and PyTorch paradigms.

## A complete Triton kernel: vector addition end to end
The following corresponds to Listing 2.1 mapping the entire vector addition process.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit                               
def add_kernel(x_ptr, y_ptr, z_ptr,
               n_elements,
               BLOCK_SIZE: tl.constexpr): 
    pid = tl.program_id(0)                
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0,
                                     BLOCK_SIZE)   
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets,
                mask=mask)                
    y = tl.load(y_ptr + offsets,
                mask=mask)                
    tl.store(z_ptr + offsets,
             x + y,
             mask=mask)                   

def add(x: torch.Tensor,
        y: torch.Tensor,
        BLOCK_SIZE: int = 1024):          
    assert x.is_cuda and y.is_cuda, \
        "Inputs must be on CUDA device"
    assert x.shape == y.shape, \
        "Input shapes must match"

    z = torch.empty_like(x)
    n_elements = x.numel()

    grid = lambda meta: (                 
        triton.cdiv(n_elements,
                    meta['BLOCK_SIZE']),
    )

    add_kernel[grid](x, y, z,
                     n_elements,
                     BLOCK_SIZE=BLOCK_SIZE) 
    return z

# Execution and Validation
torch.manual_seed(0)
size = 98432
x = torch.rand(size,
               device="cuda",
               dtype=torch.float32)
y = torch.rand(size,
               device="cuda",
               dtype=torch.float32)

output_torch = x + y
output_triton = add(x, y)

max_diff = torch.max(
    torch.abs(output_torch - output_triton)
).item()

print(f"Maximum absolute difference: {max_diff:.3e}")
assert torch.allclose(output_triton, output_torch)
print("OK — Triton result matches PyTorch.")

## Why naive timing with `time.time()` fails
A Triton kernel launch is asynchronous: the CPU queues the launch on the GPU and moves on to the next line before the kernel finishes, so wrapping the call in `time.time()` mostly measures Python dispatch overhead, not GPU execution time. The cell below shows the naive (broken) measurement, followed by the fix: bracket the call with `torch.cuda.synchronize()` so the wall clock only starts/stops once the GPU is actually idle.

In [ ]:
import time

start = time.time()
z = add(x, y)
end = time.time()
print(f"elapsed: {end - start:.6f} s")

In [ ]:
import time
import torch

torch.cuda.synchronize()
start = time.time()
z = add(x, y)
torch.cuda.synchronize()
end = time.time()
print(f"elapsed: {end - start:.6f} s")

## Correctness before performance
Before any timing number carries meaning, the kernel's output has to be checked again with a tool that reports *what* went wrong, not just whether it did. `triton.testing.assert_close` compares a result tensor against a reference within an absolute/relative tolerance and raises a descriptive error on mismatch, unlike the single boolean returned by `torch.allclose`.

In [ ]:
import torch
import triton

x = torch.rand(98432, device="cuda", dtype=torch.float32)
y = torch.rand_like(x)

reference = x + y             # trusted PyTorch result
result = add(x, y)             # our Triton kernel from earlier in this notebook

triton.testing.assert_close(result, reference)
print("Triton output matches the reference within tolerance.")

## Benchmarking the kernel
Next we benchmark pure Python natively against the optimized CUDA kernels natively accessed by `PyTorch` and our hand-built `Triton` kernel.

In [ ]:
import time
import os

@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],
        x_vals=[2**i for i in range(10, 25)],
        x_log=True,
        line_arg='provider',
        line_vals=['python', 'torch', 'triton'],
        line_names=['Python (CPU loop)', 'PyTorch (GPU)', 'Triton (GPU)'],
        styles=[('red', '-'), ('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='vector_addition_performance',
        args={},
    )
)
def benchmark(size, provider):
    quantiles = [0.5, 0.2, 0.8]
    x_gpu = torch.rand(size, device='cuda', dtype=torch.float32)
    y_gpu = torch.rand_like(x_gpu)

    if provider == 'python':
        x_cpu = x_gpu.cpu().tolist()
        y_cpu = y_gpu.cpu().tolist()
        def python_add():
            return [a + b for a, b in zip(x_cpu, y_cpu)]
        # Warmup: run a few times to reach steady state before measuring
        for _ in range(5):
            python_add()
        # Wall-clock timing: triton.testing.do_bench uses CUDA events,
        # which return near-zero for CPU-only code and inflate the GB/s figure.
        n_reps = 20
        times_ms = []
        for _ in range(n_reps):
            t0 = time.perf_counter()
            python_add()
            times_ms.append((time.perf_counter() - t0) * 1e3)
        times_ms.sort()
        ms     = times_ms[n_reps // 2]           # p50
        min_ms = times_ms[int(n_reps * 0.2)]     # p20
        max_ms = times_ms[int(n_reps * 0.8)]     # p80

    elif provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: x_gpu + y_gpu, quantiles=quantiles)

    elif provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: add(x_gpu, y_gpu), quantiles=quantiles)

    gbps = lambda ms: 12 * size / (ms * 1e-3) / 1e9
    return gbps(ms), gbps(min_ms), gbps(max_ms)

# Run the benchmark suite
os.makedirs('./vec_add_perf', exist_ok=True)
benchmark.run(print_data=True, show_plots=True, save_path='./vec_add_perf')

## Inspecting kernel output
Once the JIT triggers its target context operations, Triton compiles them into a sequence of abstract syntax tree transformations. Here we generate and inspect the IR formats.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def add_kernel_ir(x_ptr, y_ptr, z_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(z_ptr + offsets, x + y, mask=mask)

x_test = torch.rand(1024, device="cuda")
y_test = torch.rand(1024, device="cuda")
out_test = torch.empty_like(x_test)
compiled = add_kernel_ir[(1,)](x_test, y_test, out_test, 1024, BLOCK_SIZE=1024)

with open("ttir.md", "w") as f:
    f.write(compiled.asm["ttir"])

with open("ttgir.md", "w") as f:
    f.write(compiled.asm["ttgir"])

with open("llir.md", "w") as f:
    f.write(compiled.asm["llir"])

with open("ptx.md", "w") as f:
    f.write(compiled.asm["ptx"])

print("Triton IR files saved successfully.")